In [1]:
from utils.std_model import base_model

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tracers import ConsoleCallbackHandler
from langchain_core.runnables import RunnableParallel, Runnable, RunnablePassthrough

chatLLM = base_model()

summarize_chain: Runnable = (
        ChatPromptTemplate.from_messages([
            ("system", "简洁地总结以下主题："),
            ("user", "{topic}"),
        ])
        | chatLLM
        | StrOutputParser()
)

questions_chain: Runnable = (
        ChatPromptTemplate.from_messages([
            ("system", "生成关于以下主题的三个有趣问题："),
            ("user", "{topic}"),
        ])
        | chatLLM
        | StrOutputParser()
)

terms_chain: Runnable = (
        ChatPromptTemplate.from_messages([
            ("system", "从以下主题中识别 5-10 个关键术语，用逗号分隔："),
            ("user", "{topic}"),
        ])
        | chatLLM
        | StrOutputParser()
)

map_chain = RunnableParallel({
    "summary": summarize_chain,
    "questions": questions_chain,
    "terms": terms_chain,
    "topic": RunnablePassthrough(),
})

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """基于以下信息：
    摘要：{summary}
    相关问题：{questions}
    关键术语：{terms}
    综合一个全面的答案。
    """),
    ("user", "原始主题：{topic}"),
])

full_parallel_chain = map_chain | synthesis_prompt | chatLLM | StrOutputParser()

full_parallel_chain.invoke(input="太空探索的历史", config={"callbacks": [ConsoleCallbackHandler()]})


[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "太空探索的历史"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<summary,questions,terms,topic>] Entering Chain run with input:
{
  "input": "太空探索的历史"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<summary,questions,terms,topic> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "太空探索的历史"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<summary,questions,terms,topic> > chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "input": "太空探索的历史"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<summary,questions,terms,topic> > chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<summary,questions,terms,topic> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "太空探索的历史"
}
[chain

'太空探索的历史是一部从冷战竞赛到商业航天的宏大叙事。20世纪中期，美苏两极争霸催生了太空竞赛：1957年苏联率先发射“斯普特尼克1号”人造卫星，标志着人类进入太空时代；1961年加加林成为首位进入太空的人类，而美国则在1969年通过阿波罗计划实现首次载人登月。这一时期的火箭技术突飞猛进，但背后也隐藏着军事与科学的博弈——例如美苏曾计划在月球或太空进行“太空核爆炸”（如美国“A119计划”和苏联类似提案），最终因1967年《外层空间条约》禁止太空核武器，以及担心核爆会污染月球科学价值而流产。\n\n阿波罗计划之后，如果美国没有放弃载人登月，而是按原计划在1970年代建立月球基地，可能会彻底改变后续技术路线：空间站与航天飞机的开发或许会服务于地月运输，提前催生“太空酒店”、月球采矿等经济形态，甚至加速国际空间站的合作模式。但现实中，探索转向了轨道科研——苏联的和平号空间站、美国的航天飞机，以及多国共建的国际空间站，成为长期空间驻留的平台。同时，深空探测不断拓展：火星车在红色星球上巡视，旅行者号则飞向星际空间。\n\n21世纪以来，商业航天异军突起，SpaceX等公司降低了发射成本，探索目标重新聚焦月球和火星。值得注意的是，太空环境对人类行为也提出了新挑战：1965年苏联上升2号任务中，宇航员列昂诺夫出舱后因宇航服膨胀险些无法返回，这场“第一次太空犯罪”的雏形事件（如因技术故障而非故意杀人）引发了人们对太空法律与心理学的思考——尽管后来关于“宇航员因嫉妒修改轨道计算机”的说法未经证实，但它折射出极端环境下人性与规则的脆弱。\n\n从最初的卫星、火箭，到空间站、航天飞机，再到对月球和火星的持续探索，太空探索的历史不仅是技术的台阶，更是政治、伦理与冒险精神的交织。如今，国际空间站仍在轨道运行，而人类正站在重返月球、登陆火星的新起点上。'

In [2]:
from typing import TypedDict

from langgraph.constants import START, END
from langgraph.graph import StateGraph

from utils.std_model import base_model

llm = base_model()


class JokeState(TypedDict):
    topic: str
    joke: str
    story: str
    poem: str
    combined_output: str


def call_llm1(state: JokeState):
    """ LLM call to generate initial joke """
    msg = llm.invoke(f"Write a joke about {state['topic']}")
    return {
        "joke": msg.content,
    }


def call_llm2(state: JokeState):
    """ LLM call to generate story """
    msg = llm.invoke(f"Write a story about {state['topic']}")
    return {
        "story": msg.content
    }


def call_llm3(state: JokeState):
    """ LLM call to generate poem """
    msg = llm.invoke(f"Write a poem about {state['topic']}")
    return {
        "poem": msg.content
    }


def aggregator(state: JokeState):
    """ Combine the joke, story and poem into a single output """
    combined = f"Here's a story, joke, and poem about {state['topic']}!\n\n"
    combined += f"STORY:\n{state['story']}\n\n"
    combined += f"JOKE:\n{state['joke']}\n\n"
    combined += f"POEM:\n{state['poem']}"
    return {
        "combined_output": combined
    }


graph_builder = StateGraph(JokeState)

# node
graph_builder.add_node(call_llm1.__name__, call_llm1)
graph_builder.add_node(call_llm2.__name__, call_llm2)
graph_builder.add_node(call_llm3.__name__, call_llm3)
graph_builder.add_node(aggregator.__name__, aggregator)

# edge
graph_builder.add_edge(START, call_llm1.__name__)
graph_builder.add_edge(START, call_llm2.__name__)
graph_builder.add_edge(START, call_llm3.__name__)
graph_builder.add_edge(call_llm1.__name__, aggregator.__name__)
graph_builder.add_edge(call_llm2.__name__, aggregator.__name__)
graph_builder.add_edge(call_llm3.__name__, aggregator.__name__)
graph_builder.add_edge(aggregator.__name__, END)

graph = graph_builder.compile()

graph.invoke({
    "topic": "cat"
})




{'topic': 'cat',
 'joke': "Why don't cats play poker in the wild?  \nToo many cheetahs.",
 'story': 'The world was a symphony of scents. Jasper knew this better than most, for he was a cat, and his nose was his truest compass. His territory was a small, perfectly ordered universe: the sun-warmed flagstones of the garden path, the damp earth under the rose bushes that smelled of green and forgotten things, and the sprawling, leafy kingdom of the old oak tree. His human, a woman whose smell he knew as a blend of old paper and chamomile tea, called him "her little philosopher." Jasper was a tabby of considerable girth and even greater dignity, and he felt the title was apt.\n\nHis days were a ritual of quiet observation. Mornings were for the sunbeam on the rug, a golden pool of liquid warmth that he would pour himself into. Afternoons were for the garden, where he would watch bees bumble into foxgloves and listen to the scratchy murmur of the robin who lived in the hedge. The robin was a